# TREND ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("TREND ANALYSIS OF CLUSTERS")
# Load data with features
ml_data = pd.read_csv('data/clinical_report_clean.csv')
print(f"Patients loaded: {len(ml_data)}")
print(f"Available columns: {ml_data.columns.tolist()[:10]}...")

# Check if trends exist
if 'sbp_trend' not in ml_data.columns:
    print("\nTrends not found in data. Loading from original dataset...")
    
    # Load original dataset with trends
    with open('data/ts_prepared_data.pkl', 'rb') as f:
        ts_data = pickle.load(f)
    
    clustering_series = ts_data['clustering_series']
    
    # Calculate trends for all patients
    sbp_trends = {}
    dbp_trends = {}
    
    for patient in ml_data['patient_id'].tolist():
        if patient in clustering_series['SBP']:
            sbp_series = clustering_series['SBP'][patient]
            dbp_series = clustering_series['DBP'][patient]
            
            days = np.arange(len(sbp_series))
            if len(sbp_series) >= 3:
                sbp_trends[patient] = np.polyfit(days, sbp_series, 1)[0]
                dbp_trends[patient] = np.polyfit(days, dbp_series, 1)[0]
            else:
                sbp_trends[patient] = 0
                dbp_trends[patient] = 0
        else:
            sbp_trends[patient] = 0
            dbp_trends[patient] = 0
    
    # Add to dataframe
    ml_data['sbp_trend'] = ml_data['patient_id'].map(sbp_trends)
    ml_data['dbp_trend'] = ml_data['patient_id'].map(dbp_trends)
    
    print(f"Trends added for {len(ml_data)} patients")

### 1. TREND ANALYSIS BY CLUSTER

In [ ]:
cluster_trends = {}

for cluster_id in [1, 2, 3]:
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id}")
    print(f"{'='*50}")
    
    cluster_data = ml_data[ml_data['cluster'] == cluster_id]
    
    sbp_trends = cluster_data['sbp_trend'].dropna()
    dbp_trends = cluster_data['dbp_trend'].dropna()
    
    print(f"SBP trend: mean = {sbp_trends.mean():.4f} ± {sbp_trends.std():.4f}")
    print(f"DBP trend: mean = {dbp_trends.mean():.4f} ± {dbp_trends.std():.4f}")
    
    sbp_direction = "↑ increasing" if sbp_trends.mean() > 0 else "↓ decreasing"
    dbp_direction = "↑ increasing" if dbp_trends.mean() > 0 else "↓ decreasing"
    
    print(f"Direction: SBP {sbp_direction}, DBP {dbp_direction}")
    
    cluster_trends[cluster_id] = {
        'sbp_mean': sbp_trends.mean(),
        'sbp_std': sbp_trends.std(),
        'dbp_mean': dbp_trends.mean(),
        'dbp_std': dbp_trends.std()
    }

### 2. STATISTICAL COMPARISON OF TRENDS

In [ ]:
for c1, c2 in [(1,2), (1,3), (2,3)]:
    p1 = ml_data[ml_data['cluster'] == c1]['sbp_trend'].dropna()
    p2 = ml_data[ml_data['cluster'] == c2]['sbp_trend'].dropna()
    if len(p1) > 0 and len(p2) > 0:
        t_stat, p_val = stats.ttest_ind(p1, p2)
        print(f"Cluster {c1} vs {c2}: t={t_stat:.3f}, p={p_val:.4f} {'significant' if p_val < 0.05 else 'not significant'}")

### 3. VISUALIZATION

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['red', 'green', 'blue']
for idx, cluster_id in enumerate([1, 2, 3]):
    data = ml_data[ml_data['cluster'] == cluster_id]['sbp_trend'].dropna()
    axes[0].hist(data, bins=30, alpha=0.5, color=colors[idx], label=f'Cluster {cluster_id}', density=True)
axes[0].axvline(x=0, color='black', linestyle='--')
axes[0].set_xlabel('SBP Trend')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of SBP Trends')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for idx, cluster_id in enumerate([1, 2, 3]):
    data = ml_data[ml_data['cluster'] == cluster_id]['dbp_trend'].dropna()
    axes[1].hist(data, bins=30, alpha=0.5, color=colors[idx], label=f'Cluster {cluster_id}', density=True)
axes[1].axvline(x=0, color='black', linestyle='--')
axes[1].set_xlabel('DBP Trend')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of DBP Trends')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

trend_data = pd.DataFrame({
    'Cluster': ml_data['cluster'],
    'SBP Trend': ml_data['sbp_trend'],
    'DBP Trend': ml_data['dbp_trend']
})
trend_melted = trend_data.melt(id_vars=['Cluster'], var_name='Parameter', value_name='Trend')
sns.boxplot(x='Cluster', y='Trend', hue='Parameter', data=trend_melted, ax=axes[2])
axes[2].axhline(y=0, color='black', linestyle='--')
axes[2].set_title('Comparison of Trends by Cluster')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/photo/trend_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/photo/trend_analysis.png")

print("\n" + "="*60)
print("4. RECOMMENDATIONS")
print("="*60)

recommendations = pd.DataFrame({
    'Cluster': [1, 2, 3],
    'Risk': ['High', 'Medium', 'Low'],
    'SBP Trend': [
        f"{'↑' if cluster_trends[1]['sbp_mean'] > 0 else '↓'} {abs(cluster_trends[1]['sbp_mean']):.4f}",
        f"{'↑' if cluster_trends[2]['sbp_mean'] > 0 else '↓'} {abs(cluster_trends[2]['sbp_mean']):.4f}",
        f"{'↑' if cluster_trends[3]['sbp_mean'] > 0 else '↓'} {abs(cluster_trends[3]['sbp_mean']):.4f}"
    ],
    'Variability': [
        f"{cluster_trends[1]['sbp_std']:.4f}",
        f"{cluster_trends[2]['sbp_std']:.4f}",
        f"{cluster_trends[3]['sbp_std']:.4f}"
    ],
    'Recommendation': [
        'Intensify therapy, monthly monitoring',
        'Maintain therapy, control every 2-3 months',
        'Preventive observation, control every 6 months'
    ]
})

print(recommendations.to_string(index=False))
recommendations.to_csv('data/trend_recommendations.csv', index=False)
print("Saved: data/trend_recommendations.csv")

print("\n" + "="*60)
print("5. CLINICAL INTERPRETATION")
print("="*60)

for cluster_id in [1, 2, 3]:
    print(f"\nCLUSTER {cluster_id}:")
    sbp = cluster_trends[cluster_id]['sbp_mean']
    dbp = cluster_trends[cluster_id]['dbp_mean']
    
    if cluster_id == 1:
        if sbp > 0:
            print("  Progressive hypertension - therapy intensification required")
        else:
            print("  Positive dynamics - therapy is effective")
        print(f"  Visit frequency: monthly")
    
    elif cluster_id == 2:
        if abs(sbp) < 0.001:
            print("  Stable condition - maintain therapy")
        else:
            print(f"  Unstable trend ({sbp:.4f}) - monitoring required")
        print(f"  Visit frequency: every 2-3 months")
    
    else:
        if abs(sbp) < 0.001:
            print("  Optimal condition - preventive care")
        else:
            print(f"  Minor fluctuations ({sbp:.4f}) - observation")
        print(f"  Visit frequency: every 6 months")
    
    print(f"  Values: SBP trend = {sbp:.4f}, DBP trend = {dbp:.4f}")

# TESTING ALTERNATIVE HYPOTHESES
(based on trend analysis)

In [ ]:
from scipy.stats import f_oneway, shapiro, kstest

### 1. NORMALITY TEST (Shapiro-Wilk test)

In [ ]:
for cluster_id in [1, 2, 3]:
    sbp_trend = ml_data[ml_data['cluster'] == cluster_id]['sbp_trend'].dropna()
    if len(sbp_trend) < 5000:
        stat, p = shapiro(sbp_trend)
        print(f"  Cluster {cluster_id}: p={p:.4f} {'normal' if p>0.05 else 'not normal'}")
    else:
        print(f"  Cluster {cluster_id}: sample >5000, using Kolmogorov-Smirnov test")
        data_norm = (sbp_trend - sbp_trend.mean()) / sbp_trend.std()
        stat, p = kstest(data_norm, 'norm')
        print(f"    p={p:.4f} {'normal' if p>0.05 else 'not normal'}")

### 2. Alternative hypotheses for SBP trend
H₁: SBP trends differ between clusters

In [ ]:
# ANOVA for all three clusters
sbp_all = [ml_data[ml_data['cluster'] == c]['sbp_trend'].dropna() for c in [1,2,3]]
f_stat_sbp, p_anova_sbp = f_oneway(*sbp_all)
print(f"   ANOVA (all clusters): F={f_stat_sbp:.3f}, p={p_anova_sbp:.6f} → {'differences exist' if p_anova_sbp<0.05 else 'no differences'}")

# Bonferroni correction
alpha = 0.05
n_comparisons = 3
bonferroni_alpha = alpha / n_comparisons
print(f"\n   Bonferroni correction: α = {bonferroni_alpha:.4f}")

comparisons = [(1,2), (1,3), (2,3)]
print("\n   Pairwise comparisons:")
for c1, c2 in comparisons:
    p1 = ml_data[ml_data['cluster'] == c1]['sbp_trend'].dropna()
    p2 = ml_data[ml_data['cluster'] == c2]['sbp_trend'].dropna()
    t_stat, p_val = stats.ttest_ind(p1, p2)
    significant = p_val < bonferroni_alpha
    print(f"     Cluster {c1} vs {c2}: t={t_stat:.3f}, p={p_val:.5f} → {'significant' if significant else 'not significant'}")
    if c1==2 and c2==3:
        p_val_2vs3_sbp = p_val

### 3. Alternative hypotheses for DBP trend
H₁: DBP trends differ between clusters

In [ ]:
dbp_all = [ml_data[ml_data['cluster'] == c]['dbp_trend'].dropna() for c in [1,2,3]]
f_stat_dbp, p_anova_dbp = f_oneway(*dbp_all)
print(f"   ANOVA: F={f_stat_dbp:.3f}, p={p_anova_dbp:.6f} → {'differences exist' if p_anova_dbp<0.05 else 'no differences'}")

for c1, c2 in comparisons:
    p1 = ml_data[ml_data['cluster'] == c1]['dbp_trend'].dropna()
    p2 = ml_data[ml_data['cluster'] == c2]['dbp_trend'].dropna()
    t_stat, p_val = stats.ttest_ind(p1, p2)
    significant = p_val < bonferroni_alpha
    print(f"     Cluster {c1} vs {c2}: t={t_stat:.3f}, p={p_val:.5f} → {'significant' if significant else 'not significant'}")

### 4. Alternative hypotheses for SBP variability
H₁: SBP variability differs between clusters

In [ ]:
std_all = [ml_data[ml_data['cluster'] == c]['SBP_std'].dropna() for c in [1,2,3]]
f_stat_std, p_anova_std = f_oneway(*std_all)
print(f"   ANOVA: F={f_stat_std:.3f}, p={p_anova_std:.6f} → {'differences exist' if p_anova_std<0.05 else 'no differences'}")

for c1, c2 in comparisons:
    p1 = ml_data[ml_data['cluster'] == c1]['SBP_std'].dropna()
    p2 = ml_data[ml_data['cluster'] == c2]['SBP_std'].dropna()
    t_stat, p_val = stats.ttest_ind(p1, p2)
    significant = p_val < bonferroni_alpha
    print(f"     Cluster {c1} vs {c2}: t={t_stat:.3f}, p={p_val:.5f} → {'significant' if significant else 'not significant'}")

### 5. Test: variability in cluster 2 is higher than in cluster 3

In [ ]:
std2 = ml_data[ml_data['cluster'] == 2]['SBP_std'].dropna()
std3 = ml_data[ml_data['cluster'] == 3]['SBP_std'].dropna()
t_stat, p_val_var = stats.ttest_ind(std2, std3)
print(f"   Cluster 2 mean = {std2.mean():.4f}, Cluster 3 mean = {std3.mean():.4f}")
print(f"   t-test: p={p_val_var:.5f} → {'variability in cluster 2 is HIGHER (significant)' if p_val_var<0.05 and std2.mean() > std3.mean() else 'not confirmed'}")

### FINAL CONCLUSIONS ON ALTERNATIVE HYPOTHESES

In [ ]:
hypotheses = {
    "H₁: SBP trends differ between clusters": p_anova_sbp < 0.05,
    "H₁: DBP trends differ between clusters": p_anova_dbp < 0.05,
    "H₁: SBP variability differs between clusters": p_anova_std < 0.05,
    "H₁: Cluster 2 differs from cluster 3 by SBP trend": p_val_2vs3_sbp < bonferroni_alpha,
    "H₁: SBP variability in cluster 2 is higher than in cluster 3": (p_val_var < 0.05) and (std2.mean() > std3.mean())
}

for h, is_true in hypotheses.items():
    status = "CONFIRMED" if is_true else "NOT CONFIRMED"
    print(f"  {h}: {status}")